# Factors affecting the gut microbiota in infants

Summary: Analysis of the temporal volatility of the gut microbiota in infants per age/timepoint, and the effects of sleep and feeding history on the gut microbiota.

Author, date: Fannie Kerff, January-August 2025

Environment: qiime2-amplicon-2024.10 (saved in `environment.yml`)

## Setup

In [ ]:
# imports
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

from functions_script import load_tsv, scatterplot, scatterplot_age, plot_estimates

%matplotlib inline

In [ ]:
# paths
# IF ACCESS TO ALL STUDY DATA
path = "/cluster/work/bokulich/fkerff/GrumpyBiome-analysis"
meta_data_path = f"{path}/data-sensitive/meta_data"
figures_path = f"{path}/figures"

# ELSE (ONLY ACCESS TO PUBLIC DATA)
# meta_data_path = "../data/meta_data"
# figures_path = "../figures"

In [ ]:
# create color palette
colors = [plt.cm.Spectral(i/float(6)) for i in range(7)]
timepoints_colors = [colors[0], colors[5], colors[6]]

## Load infants metadata

In [ ]:
# load samples metadata
md_samples = load_tsv(f"{meta_data_path}/md_samples.tsv")
md_samples = md_samples.sort_values(by=["timepoint", "infant_id"])

print(md_samples.shape)
md_samples.head()

In [ ]:
# load infant metadata per age/timepoint
md_ages = load_tsv(f"{meta_data_path}/md_ages.tsv")
md_ages = md_ages.sort_values(by=["timepoint", "infant_id"])

print(md_ages.shape)
md_ages.head()

## Microbiota temporal volatility per age/timepoint

In [ ]:
# create data for volatility analysis
data_volatility = md_ages.dropna(subset="volatility_braycurtis")
print(data_volatility.shape)

In [ ]:
# linear mixed effects model for temporal volatility measured via Bray-Curtis dissimilarity
model_bc = smf.mixedlm("volatility_braycurtis ~ age_days + sex",
                    data_volatility, 
                    groups=data_volatility["infant_id"]).fit()
print(model_bc.summary())

In [ ]:
# linear mixed effects model for temporal volatility measured via Jaccard dissimilarity
model_ja = smf.mixedlm("volatility_jaccard ~ age_days + sex",
                    data_volatility, 
                    groups=data_volatility["infant_id"]).fit()
print(model_ja.summary())

In [ ]:
# linear mixed effects model for temporal volatility measured via unweighted UniFrac distance
model_uu = smf.mixedlm("volatility_unweighted_unifrac ~ age_days + sex",
                    data_volatility, 
                    groups=data_volatility["infant_id"]).fit()
print(model_uu.summary())

In [ ]:
# linear mixed effects model for temporal volatility measured via weighted UniFrac distance
model_wu = smf.mixedlm("volatility_weighted_unifrac ~ age_days + sex",
                    data_volatility, 
                    groups=data_volatility["infant_id"]).fit()
print(model_wu.summary())

In [ ]:
# plot age vs. volatility
fig, axs = plt.subplots(1, 4, figsize=(16, 6))

scatterplot(data_volatility, "age_days", "volatility_braycurtis", "timepoint", 
                    "", "Temporal volatility: Bray-Curtis dissimilarity", "Age (days)",
                    timepoints_colors, axs[0], loc_position = 'upper right')

scatterplot(data_volatility, "age_days", "volatility_jaccard", "timepoint", 
                    "", "Temporal volatility: Jaccard similarity index", "Age (days)",
                    timepoints_colors, axs[1], loc_position = None)

scatterplot(data_volatility, "age_days", "volatility_unweighted_unifrac", "timepoint", 
                    "", "Temporal volatility: Unweighted UniFrac distance", "Age (days)",
                    timepoints_colors, axs[2], loc_position = None)

scatterplot(data_volatility, "age_days", "volatility_weighted_unifrac", "timepoint", 
                    "", "Temporal volatility: Weighted UniFrac distance", "Age (days)",
                    timepoints_colors, axs[3], loc_position = None)

plt.tight_layout()

plt.savefig(f"{figures_path}/age-and-volatility.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot age vs. unweighted unifrac volatility
fig, axs = plt.subplots(1, 1, figsize=(4, 6))

scatterplot(data_volatility, "age_days", "volatility_unweighted_unifrac", "timepoint", 
                    "", "Temporal volatility: Unweighted UniFrac distance", "Age (days)",
                    timepoints_colors, ax=axs, loc_position = 'upper right')

plt.savefig(f"{figures_path}/volatility_unweighted_unifrac_age.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# coefficient plots
models = [model_bc, model_ja, model_uu, model_wu]
titles = ["Temporal volatility: \n Bray-Curtis dissimilarity", 
          "Temporal volatility: \n Jaccard similarity index", 
          "Temporal volatility: \n  Unweighted UniFrac distance", 
          "Temporal volatility: \n Weighted UniFrac distance"]
x_pos = [0.02, 0.01, 0.01, 0.01]
plot_estimates(models, titles, x_pos, f"{figures_path}/volatility_estimates.pdf")

## Effects of sleep and feeding history on gut microbiota diversity

In [ ]:
# define factor columns
cols = ["time_since_last_feeding_in_h", 
        "prior_time_spent_awake_in_h", 
        "duration_of_last_sleep_in_h"]

In [ ]:
# extract data for factors analysis
data_factors = md_samples.dropna(subset=cols)
print(data_factors.shape)

In [ ]:
# gut microbiota diversity over age
fig, axs = plt.subplots(1, 4, figsize=(16, 6))

scatterplot(data_factors, "age_days", "observed_features", "timepoint", 
                    "", "Alpha diversity: Observed features", "Age (days)",
                    timepoints_colors, axs[0], loc_position = "upper center")

scatterplot(data_factors, "age_days", "shannon_entropy", "timepoint", 
                    "", "Alpha diversity: Shannon entropy", "Age (days)",
                    timepoints_colors, axs[1], loc_position=None)

scatterplot(data_factors, "age_days", "pielou_evenness", "timepoint", 
                    "", "Alpha diversity: Pielou evenness", "Age (days)",
                    timepoints_colors, axs[2], loc_position=None)

scatterplot(data_factors, "age_days", "faith_pd", "timepoint", 
                    "", "Alpha diversity: Faith phylogenetic diversity", "Age (days)",
                    timepoints_colors, axs[3], loc_position=None)

plt.tight_layout()

plt.savefig(f"{figures_path}/alpha_div_scatter.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# linear mixed effects model for observed features
model_of = smf.mixedlm("observed_features ~ time_since_last_feeding_in_h + prior_time_spent_awake_in_h + duration_of_last_sleep_in_h + age_days + sex",
                    data=data_factors,
                    groups=data_factors["infant_id"]).fit()
print(model_of.summary())

In [ ]:
# linear mixed effects model for shannon entropy
model_sh = smf.mixedlm("shannon_entropy ~ time_since_last_feeding_in_h + prior_time_spent_awake_in_h + duration_of_last_sleep_in_h + age_days + sex",
                    data=data_factors,
                    groups=data_factors["infant_id"]).fit()
print(model_sh.summary())

In [ ]:
# linear mixed effects model for pielou evenness
model_pi = smf.mixedlm("pielou_evenness ~ time_since_last_feeding_in_h + prior_time_spent_awake_in_h + duration_of_last_sleep_in_h + age_days + sex",
                    data=data_factors,
                    groups=data_factors["infant_id"]).fit()
print(model_pi.summary())

In [ ]:
# linear mixed effects model for faith_pd
model_fa = smf.mixedlm("faith_pd ~ time_since_last_feeding_in_h + prior_time_spent_awake_in_h + duration_of_last_sleep_in_h + age_days + sex",
                    data=data_factors,
                    groups=data_factors["infant_id"]).fit()
print(model_fa.summary())

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(16, 6))

scatterplot(data_factors, "age_days", "observed_features", "timepoint", 
                "", "Alpha diversity: Observed features", "Age (days)",
                palette=timepoints_colors, ax=axs[0], loc_position="upper left")

scatterplot_age(data_factors, "prior_time_spent_awake_in_h", "observed_features", "timepoint",
                "", "", "Prior time spent awake (h)",
                palette=timepoints_colors, ax=axs[1], loc_position=None)

scatterplot_age(data_factors, "duration_of_last_sleep_in_h", "observed_features", "timepoint", 
                    "", "", "Duration of the last sleep (h)",
                    palette=timepoints_colors, ax=axs[2], loc_position=None)

scatterplot_age(data_factors, "time_since_last_feeding_in_h", "observed_features", "timepoint", "",
                "", "Time since the last feeding (h)", 
                palette=timepoints_colors, ax=axs[3], loc_position=None)

plt.tight_layout()

plt.savefig(f"{figures_path}/obs_feat_scatter_factors.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# coefficient plots
models = [model_of, model_sh, model_pi, model_fa]
titles = ["Alpha diversity: \n Observed features", 
          "Alpha diversity: \n Shannon entropy", 
          "Alpha diversity: \n Pielou evenness", 
          "Alpha diversity: \n Faith's phylogenetic diversity"]
x_pos = [-4, 0.1, 0.02, -0.4]
plot_estimates(models, titles, x_pos, f"{figures_path}/factors_estimates.pdf")